In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import datetime as dt

In [25]:
rfm_data = pd.read_csv(r"C:\Users\moham\Downloads\rfm_data.csv")
rfm_data

,CustomerID,PurchaseDate,TransactionAmount,ProductInformation,OrderID,Location
0,8814,2023-04-11,943.31,Product C,890075,Tokyo
1,2188,2023-04-11,463.70,Product A,176819,London
2,4608,2023-04-11,80.28,Product A,340062,New York
3,2559,2023-04-11,221.29,Product A,239145,London
4,9482,2023-04-11,739.56,Product A,194545,Paris
...,...,...,...,...,...,...
995,2970,2023-06-10,759.62,Product B,275284,London
996,6669,2023-06-10,941.50,Product C,987025,New York
997,8836,2023-06-10,545.36,Product C,512842,London
998,1440,2023-06-10,729.94,Product B,559753,Paris


In [27]:
## 
rfm_data.duplicated().sum()

0

In [29]:
rfm_data.isnull().sum()

CustomerID            0
PurchaseDate          0
TransactionAmount     0
ProductInformation    0
OrderID               0
Location              0
dtype: int64

In [31]:
## 
rfm_data.dtypes

CustomerID              int64
PurchaseDate           object
TransactionAmount     float64
ProductInformation     object
OrderID                 int64
Location               object
dtype: object

In [33]:
 rfm_data["PurchaseDate"] = pd.to_datetime(rfm_data["PurchaseDate"])

In [35]:
rfm_data.dtypes

CustomerID                     int64
PurchaseDate          datetime64[ns]
TransactionAmount            float64
ProductInformation            object
OrderID                        int64
Location                      object
dtype: object

In [ ]:
$selctin the columns needed for rfm
# recency - PurchaseDate          
# monetary - TransactionAmoun
#frequency - OrderID

In [37]:
rfm_data = rfm_data[['CustomerID','PurchaseDate' , 'TransactionAmount' ,'OrderID' ]]
rfm_data

,CustomerID,PurchaseDate,TransactionAmount,OrderID
0,8814,2023-04-11,943.31,890075
1,2188,2023-04-11,463.70,176819
2,4608,2023-04-11,80.28,340062
3,2559,2023-04-11,221.29,239145
4,9482,2023-04-11,739.56,194545
...,...,...,...,...
995,2970,2023-06-10,759.62,275284
996,6669,2023-06-10,941.50,987025
997,8836,2023-06-10,545.36,512842
998,1440,2023-06-10,729.94,559753


In [41]:
## renaming the columns to rfm
rfm_data.rename(columns = {'PurchaseDate' : 'Recency' ,'TransactionAmount' :'Monetary' , "OrderID" :'Frequency'},inplace=True)
rfm_data

,CustomerID,Recency,Monetary,Frequency
0,8814,2023-04-11,943.31,890075
1,2188,2023-04-11,463.70,176819
2,4608,2023-04-11,80.28,340062
3,2559,2023-04-11,221.29,239145
4,9482,2023-04-11,739.56,194545
...,...,...,...,...
995,2970,2023-06-10,759.62,275284
996,6669,2023-06-10,941.50,987025
997,8836,2023-06-10,545.36,512842
998,1440,2023-06-10,729.94,559753


In [45]:
max_date = rfm_data['Recency'].max()
max_date

Timestamp('2023-06-10 00:00:00')

In [51]:
latest_date = dt.datetime(2023 ,6,11)
latest_date

datetime.datetime(2023, 6, 11, 0, 0)

In [59]:
group_data = rfm_data.groupby(by = 'CustomerID').agg({'Recency' : lambda x:(latest_date-x.max()).days ,
                                                      'Monetary':lambda x : x.sum(),
                                                       'Frequency':lambda x : x.count() })

In [61]:
group_data

,Recency,Monetary,Frequency
CustomerID,,,
1011,34,1129.02,2
1025,22,359.29,1
1029,1,704.99,1
1046,44,859.82,1
1049,14,225.72,1
...,...,...,...
9941,43,960.53,1
9950,39,679.11,1
9954,13,798.01,1


In [65]:

ecom_data_rfm = group_data.copy()
ecom_data_rfm

,Recency,Monetary,Frequency
CustomerID,,,
1011,34,1129.02,2
1025,22,359.29,1
1029,1,704.99,1
1046,44,859.82,1
1049,14,225.72,1
...,...,...,...
9941,43,960.53,1
9950,39,679.11,1
9954,13,798.01,1


In [67]:
for i in  ecom_data_rfm.columns:
    print("*********",i,"*********")
    print(ecom_data_rfm[i].min())
    print(ecom_data_rfm[i].max())
        


********* Recency *********
1
61
********* Monetary *********
12.13
2379.4500000000003
********* Frequency *********
1
3


In [73]:
quantiles = ecom_data_rfm.quantile([ 0.25, 0.50, 0.75])
quantiles

quantile_dict = quantiles.to_dict()
quantile_dict

{'Recency': {0.25: 15.0, 0.5: 32.0, 0.75: 45.0},
 'Monetary': {0.25: 266.64, 0.5: 542.895, 0.75: 782.6949999999999},
 'Frequency': {0.25: 1.0, 0.5: 1.0, 0.75: 1.0}}

In [104]:
def recency_score(x,q,d):
    if x<=d[q][0.25]:
        return 1
    elif x<=d[q][0.50]:
        return 2
    elif x<=d[q][0.75]:
        return 3
    else :
        return 4
    
def freq_money_score(x,q,d):
    if x<=d[q][0.25]:
        return 4
    elif x<=d[q][0.50]:
        return 3
    elif x<=d[q][0.75]:
        return 2
    else :
        return 1
    
    
    

In [106]:
ecom_data_rfm["recency_score"] = ecom_data_rfm["Recency"].apply(recency_score,args = ('Recency',quantile_dict))

In [116]:
ecom_data_rfm["frequency_score"] = ecom_data_rfm["Frequency"].apply(freq_money_score,args = ('Frequency',quantile_dict))
ecom_data_rfm["monetory_score"] = ecom_data_rfm["Monetary"].apply(freq_money_score,args = ('Monetary',quantile_dict))

In [118]:
ecom_data_rfm

,Recency,Monetary,Frequency,recency_score,frequency_score,monetory_score
CustomerID,,,,,,
1011,34,1129.02,2,3,1,1
1025,22,359.29,1,2,4,3
1029,1,704.99,1,1,4,2
1046,44,859.82,1,3,4,1
1049,14,225.72,1,1,4,4
...,...,...,...,...,...,...
9941,43,960.53,1,3,4,1
9950,39,679.11,1,3,4,2
9954,13,798.01,1,1,4,1


In [120]:
ecom_data_rfm['loyality_Score'] = ecom_data_rfm[['recency_score' ,'frequency_score' , 'monetory_score' ]].sum(axis=1)


In [122]:
ecom_data_rfm

,Recency,Monetary,Frequency,recency_score,frequency_score,monetory_score,loyality_Score
CustomerID,,,,,,,
1011,34,1129.02,2,3,1,1,5
1025,22,359.29,1,2,4,3,9
1029,1,704.99,1,1,4,2,7
1046,44,859.82,1,3,4,1,8
1049,14,225.72,1,1,4,4,9
...,...,...,...,...,...,...,...
9941,43,960.53,1,3,4,1,8
9950,39,679.11,1,3,4,2,9
9954,13,798.01,1,1,4,1,6


In [126]:
badges = ["platinum" , "gold" , "silver" , "bronze"]
score_badge = pd.qcut(ecom_data_rfm['loyality_Score'] , q=4 ,labels=badges )
score_badge
ecom_data_rfm["loaylit_badges"] = score_badge

In [128]:
ecom_data_rfm

,Recency,Monetary,Frequency,recency_score,frequency_score,monetory_score,loyality_Score,loaylit_badges
CustomerID,,,,,,,,
1011,34,1129.02,2,3,1,1,5,platinum
1025,22,359.29,1,2,4,3,9,gold
1029,1,704.99,1,1,4,2,7,platinum
1046,44,859.82,1,3,4,1,8,platinum
1049,14,225.72,1,1,4,4,9,gold
...,...,...,...,...,...,...,...,...
9941,43,960.53,1,3,4,1,8,platinum
9950,39,679.11,1,3,4,2,9,gold
9954,13,798.01,1,1,4,1,6,platinum


In [130]:
#dropping all the scorers columns
ecom_data_rfm.drop(columns= ['recency_score' ,'frequency_score' , 'monetory_score' ,'loyality_Score' ],axis=1 , inplace=True)


In [132]:
ecom_data_rfm

,Recency,Monetary,Frequency,loaylit_badges
CustomerID,,,,
1011,34,1129.02,2,platinum
1025,22,359.29,1,gold
1029,1,704.99,1,platinum
1046,44,859.82,1,platinum
1049,14,225.72,1,gold
...,...,...,...,...
9941,43,960.53,1,platinum
9950,39,679.11,1,gold
9954,13,798.01,1,platinum


In [ ]:
ecom_data_rfm["loaylit_badges"].value_counts()